# Evaluator Agent — Code Guide

**Owner:** Sabina Rudolph  
**Target file:** `agents/sabina_evaluator.py`  

> This notebook is documentation only — do not run it as the source of truth. The real source of truth is `agents/sabina_evaluator.py`. This guide mirrors the current code and explains why each part exists, so someone reading it can understand the evaluator without having to mentally unpack the whole pipeline first.

## What this agent does

This notebook documents the Evaluator Agent, the third handoff of the stock-move prediction pipeline. It reads Nadi's `predictions_test.csv` and generated `classifier.py`, scores one requested held-out split, reviews the classifier code, and writes Jack's input file: `evaluation_report.json`.

The split part is important. During retuning, my Evaluator agent scores only the `val` rows. After Jack has selected the best iteration, my Evaluator agent the `test` rows exactly once for the final report. That keeps the loop from slowly learning the test set by accident.

In simpler words: Nadi gives me the model's guesses, and my agent grades them. It checks the overall accuracy, which labels are weak, whether a label is missing from the selected split, and whether the generated classifier code has anything obvious Jack should know about.

The evaluator's main job is to produce metrics and a proposal. The proposal helps Jack decide whether the loop should retune or proceed. My Evaluator agent recommends; Jack decides.

The current design keeps the risky parts deterministic:

- `accuracy`, `class_accuracy`, `class_support`, and misclassified ids
- `recommended_action`
- `focus_labels`
- `suggested_params`

If Ollama is enabled, the LLM only improves the human-readable `reason` and `code_notes`. It can help explain what went wrong, but it cannot quietly change the gate.


## 1. Imports and constants

These are the standard libraries and constants the evaluator needs:

- **`json`** — write Jack's `evaluation_report.json`
- **`os`** — build output paths and read environment variables
- **`re`** — inspect simple assignments inside Nadi's generated `classifier.py`
- **`sys`** — print warnings when classifier constants cannot be parsed
- **`urllib.request`** — call local Ollama without adding another dependency
- **`TypedDict`** — document the state passed through LangGraph
- **`Agent`** — the shared base class so all agents expose the same `.run()` style interface
- **`LABELS` / contract readers** — shared Handoff 2 rules from `agents/contracts.py`

The cross-agent file shape lives in `agents/contracts.py`; I should not keep my own copy of the prediction columns. The scoring policy still lives here, because that is evaluator behavior: target accuracy, threshold step size, the class-collapse floor, and the LLM settings.

Two constants are especially important now:

- `FOCUS_MARGIN = 0.05` means labels close to the weakest label are treated as weak together.
- `CLASS_COLLAPSE_FLOOR = 0.05` means a supported class with recall below 5% is treated as a collapse signal, not just a harmless low score.


In [ ]:
import json
import os
import re
import sys
import urllib.request
from typing import Callable, TypedDict

try:
    from agents.base import Agent
    from agents.contracts import (
        LABELS,
        PREDICTION_COLUMNS,
        read_prediction_rows,
        validate_prediction_rows,
    )
except ModuleNotFoundError:
    from base import Agent
    from contracts import (
        LABELS,
        PREDICTION_COLUMNS,
        read_prediction_rows,
        validate_prediction_rows,
    )

OUTPUT_DIR = "outputs"
TARGET_ACCURACY = 0.60
DEFAULT_RETUNE_THRESHOLD = 0.50
THRESHOLD_STEP = 0.05
THRESHOLD_FLOOR = 0.20
DEFAULT_MAX_LENGTH = 128
FOCUS_MARGIN = 0.05
CLASS_COLLAPSE_FLOOR = 0.05

# Read once at import time so tests and demos get stable evaluator behavior.
USE_OLLAMA = os.getenv("EVALUATOR_USE_OLLAMA", "false").lower() == "true"
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MODEL = os.getenv("EVALUATOR_OLLAMA_MODEL", "llama3.1")
OLLAMA_TIMEOUT_SECONDS = 30
LLM_TEMPERATURE = 0.2
MISCLASSIFIED_SAMPLE_SIZE = 25

PROPOSAL_FIELDS = {
    "recommended_action", "reason", "focus_labels", "suggested_params", "code_notes",
}


## 2. Evaluator state

This is the small state dictionary that LangGraph passes through my evaluator graph.

It starts with file paths and one split selector:

- `predictions_path` — where Nadi wrote `predictions_test.csv`
- `classifier_code_path` — where Nadi wrote `classifier.py`
- `output_path` — where my Evaluator agent should write `evaluation_report.json`
- `eval_split` — either `val` during retune cycles or `test` for the final report

Then the graph adds the loaded rows, the classifier source text, and finally the finished report.

I kept this state intentionally small. My Evaluator agent is not the orchestrator. My job is one handoff: read Nadi's outputs, score the requested split honestly, and write Jack's input.


In [ ]:
class EvaluatorState(TypedDict, total=False):
    predictions_path: str
    classifier_code_path: str
    output_path: str
    eval_split: str
    predictions: list[dict]
    code_text: str
    code_notes: str
    report: dict


## 3. Reading and validating the Classifier Agent's files

Before calculating anything, the evaluator checks that the Classifier Agent's CSV matches the data contract. This is important because otherwise the accuracy number could look valid while being based on the wrong columns or mixed train/test data.

The validation rules live in `agents/contracts.py`. My Evaluator agent calls those helpers instead of re-implementing them locally, because the same Handoff 2 rules are also used by Nadi's generated-code guardrail and Jack's final output path. My Evaluator agent later chooses which split to score with `eval_split`.

The validation checks:

- the columns are exactly the Handoff 2 columns
- the file is not empty
- every row is from a held-out split (`val` or `test`), never `train`
- labels and predictions are only `up`, `down`, or `neutral`
- probabilities and confidence are in the 0-1 range
- `prob_up + prob_down + prob_neutral` is approximately 1
- `confidence` equals the highest probability

This is a bit strict on purpose. If the classifier output is malformed, it is better to fail early than hand the Manager Agent a misleading report. It is basically the boring checking part, but it matters because everything after this depends on the file being correct.

In [ ]:
def _read_predictions(path: str) -> list[dict]:
    # Read classifier predictions and enforce the exact Handoff 2 columns.
    return read_prediction_rows(path)


def _read_code(path: str) -> str:
    with open(path, encoding="utf-8") as f:
        return f.read()


def validate_predictions(rows: list[dict]) -> None:
    # Check the Handoff 2 fields needed before scoring.
    validate_prediction_rows(rows)

## 4. Computing the metrics

This is the report-card part of the evaluator.

The main metric is **accuracy**, because this is a three-class classification task (`up`, `down`, `neutral`). Sabina also reports two per-class views:

- `class_accuracy`: how well the classifier did on each true label
- `class_support`: how many rows of each true label existed in the selected split

The support count fixes an important edge case. If the selected split has no `down` rows, `class_accuracy["down"]` is still `0.0` for compatibility, but that is not a model collapse. It just means there was nothing to score for that label. Jack uses `class_support` to tell those two cases apart.

The output of this function becomes the top-level metric section of `evaluation_report.json`: `accuracy`, `below_threshold`, `class_accuracy`, `class_support`, `misclassified_count`, and `misclassified_ids`.


In [ ]:
def compute_metrics(rows: list[dict]) -> dict:
    # Compute overall and per-class classification accuracy/support.
    total = len(rows)
    wrong = [row for row in rows if row["label"] != row["predicted_label"]]
    class_accuracy = {}
    class_support = {}

    for label_name in LABELS:
        class_rows = [row for row in rows if row["label"] == label_name]
        correct = sum(row["predicted_label"] == label_name for row in class_rows)
        class_support[label_name] = len(class_rows)
        class_accuracy[label_name] = (
            round(correct / len(class_rows), 2) if class_rows else 0.0
        )

    accuracy = round((total - len(wrong)) / total, 2)
    return {
        "accuracy": accuracy,
        "below_threshold": accuracy < TARGET_ACCURACY,
        "class_accuracy": class_accuracy,
        "class_support": class_support,
        "misclassified_count": len(wrong),
        "misclassified_ids": [row["article_id"] for row in wrong],
    }

## 5. Helpers — reading simple settings from `classifier.py`

Nadi's classifier is generated code, but there are a few useful constants inside it, like `THRESHOLD`, `MAX_LENGTH`, `MODEL`, and `MODEL_DIR`.

My Evaluator agent does not execute the generated classifier. It only reads the file as text and extracts simple assignments with regex. That is enough for review notes and deterministic retune suggestions.

If a numeric value is present but cannot be parsed, the evaluator prints a warning and falls back to the default. This is better than silently pretending everything is fine, because a bad generated constant can otherwise make the retune loop look stuck for mysterious reasons.


In [ ]:
def _find_assignment(code_text: str, name: str) -> str | None:
    match = re.search(rf"^\s*{re.escape(name)}\s*=\s*([^\n#]+)", code_text, re.M)
    return match.group(1).strip() if match else None


def _warn_unparseable(name: str, value: str, default: float | int) -> None:
    print(
        f"[sabina] could not parse {name}={value!r} in classifier.py; "
        f"assuming {default}",
        file=sys.stderr,
    )


def _float_assignment(code_text: str, name: str, default: float) -> float:
    value = _find_assignment(code_text, name)
    if value is None:
        return default
    try:
        return float(value.strip("\"'"))
    except ValueError:
        _warn_unparseable(name, value, default)
        return default


def _int_assignment(code_text: str, name: str, default: int) -> int:
    value = _find_assignment(code_text, name)
    if value is None:
        return default
    try:
        return int(float(value.strip("\"'")))
    except ValueError:
        _warn_unparseable(name, value, default)
        return default


## 6. Helper — finding the weakest labels

Originally it would have been easy to only pick the exact weakest class. But in real data, class accuracies almost never tie exactly, and two classes can be weak in the same practical sense.

So the current version uses a small margin: every supported class within `0.05` of the weakest supported class is included in `focus_labels`.

Example:

- `up = 0.30`
- `down = 0.28`
- `neutral = 0.45`

The exact weakest is `down`, but `up` is basically also weak. With `FOCUS_MARGIN = 0.05`, both `down` and `up` are included.

The newer detail is `class_support`: labels with support `0` are ignored when choosing focus labels. If there are no `down` rows in the selected split, my Evaluator agent should not tell Nadi to focus on `down` just because the compatibility accuracy is `0.0`.


In [ ]:
def _weakest_labels(class_accuracy: dict, class_support: dict | None = None) -> list[str]:
    supported = {
        label_name: score
        for label_name, score in class_accuracy.items()
        if class_support is None or class_support.get(label_name, 0) > 0
    }
    if not supported:
        supported = class_accuracy

    weakest_score = min(supported.values())
    return [
        label_name
        for label_name, score in supported.items()
        if score <= weakest_score + FOCUS_MARGIN
    ]


## 7. Static classifier review

This function writes short notes about the generated classifier and the metrics. It is intentionally simple and deterministic.

Right now it looks for four useful signals:

- whether the threshold is hardcoded in `classifier.py`
- whether a supported class has near-zero recall, which is a real collapse signal
- whether a label has support `0`, which is **not** a collapse
- whether the neutral class is one of the weakest supported labels

That support distinction is the newest part of my evaluator. Without it, a missing label and a collapsed label both looked like `0.0`, and Jack's gate had no way to tell the difference.

`review_classifier_code` remains as a backwards-compatible wrapper, but `build_report` now calls `review_classifier_metrics` because the better review needs both `class_accuracy` and `class_support`.


In [ ]:
def review_classifier_code(code_text: str, class_accuracy: dict) -> str:
    return review_classifier_metrics(code_text, class_accuracy)


def review_classifier_metrics(
    code_text: str,
    class_accuracy: dict,
    class_support: dict | None = None,
) -> str:
    notes = []

    threshold = _find_assignment(code_text, "THRESHOLD")
    if threshold is not None:
        notes.append(f"threshold hardcoded at {threshold} in classifier.py")

    collapsed = [
        name for name, score in class_accuracy.items()
        if (class_support is None or class_support.get(name, 0) > 0)
        and score < CLASS_COLLAPSE_FLOOR
    ]
    if collapsed:
        notes.append(
            f"class collapse: {', '.join(collapsed)} recall near zero — "
            "aggregate accuracy mostly reflects the majority class share, not signal"
        )

    missing = [
        name for name in class_accuracy
        if class_support is not None and class_support.get(name, 0) == 0
    ]
    if missing:
        notes.append(
            f"no {', '.join(missing)} rows in this eval split — support is zero, "
            "so this is not evidence of class collapse"
        )

    weakest_labels = _weakest_labels(class_accuracy, class_support)
    if "neutral" in weakest_labels:
        notes.append("the neutral band (+/-1%) may be too narrow for the neutral class")

    return "; ".join(notes)


## 8. Suggesting retune parameters

This is one of the parts where the evaluator must stay deterministic.

If accuracy is below the target, the Evaluator Agent recommends `retune`. The suggested threshold steps down by `0.05` from the classifier's current threshold, but never below `0.20`. That floor matches the Manager Agent's retune schedule.

`max_length` is read from the classifier if possible, otherwise it defaults to `128`.

The LLM is not allowed to invent these values because the Manager Agent and Classifier Agent depend on them being stable.

In [ ]:
def _suggest_retune_params(code_text: str) -> dict:
    # Suggest the next deterministic retune step without asking the LLM.
    current_threshold = _float_assignment(
        code_text, "THRESHOLD", DEFAULT_RETUNE_THRESHOLD
    )
    next_threshold = max(THRESHOLD_FLOOR, current_threshold - THRESHOLD_STEP)
    max_length = _int_assignment(code_text, "MAX_LENGTH", DEFAULT_MAX_LENGTH)

    return {
        "threshold": round(next_threshold, 2),
        "max_length": max_length,
    }

## 9. Building the deterministic base proposal

The `proposal` object is my Evaluator agents recommendation to Jack. Jack still owns the final decision.

This function builds the first version of the proposal completely without the LLM:

- if accuracy is below `0.60`, recommend `retune`
- if accuracy clears `0.60`, recommend `proceed`
- include the weakest or near-weakest **supported** labels
- include suggested params only for retune
- include a plain-English reason and code notes

This is the version we fall back to if Ollama is off, unavailable, or returns invalid JSON.

One subtle detail: the weakest score is computed only from the selected `focus_labels`, which already filtered out zero-support labels. That keeps the reason aligned with what Jack and Nadi can actually act on.


In [ ]:
def make_base_proposal(metrics: dict, code_text: str, code_notes: str) -> dict:
    focus_labels = _weakest_labels(
        metrics["class_accuracy"],
        metrics.get("class_support"),
    )
    weakest_score = min(metrics["class_accuracy"][label] for label in focus_labels)

    if metrics["below_threshold"]:
        reason = (
            f"accuracy {metrics['accuracy']:.2f} below target "
            f"{TARGET_ACCURACY:.2f}; {', '.join(focus_labels)} class weakest"
        )
        return {
            "recommended_action": "retune",
            "reason": reason,
            "focus_labels": focus_labels,
            "suggested_params": _suggest_retune_params(code_text),
            "code_notes": code_notes,
        }

    reason = (
        f"accuracy {metrics['accuracy']:.2f} clears the {TARGET_ACCURACY:.2f} "
        f"target; {', '.join(focus_labels)} class is weakest "
        f"({weakest_score:.2f}) but the iteration budget favours proceeding"
    )
    return {
        "recommended_action": "proceed",
        "reason": reason,
        "focus_labels": focus_labels,
        "suggested_params": {},
        "code_notes": code_notes,
    }


## 10. Validating the proposal

This is the safety layer. Even though the base proposal is deterministic, I still validate it, and I also validate the final proposal after any LLM text is applied.

The validator makes sure:

- all required fields exist
- no extra fields sneak in
- `recommended_action` is only `retune` or `proceed`
- `recommended_action` matches the accuracy gate
- focus labels are real labels
- `suggested_params` is a dictionary
- `suggested_params` only appear when the action is `retune`
- retune params include numeric `threshold` and integer `max_length`
- `reason` and `code_notes` have the expected text types

This is why the LLM can be useful without becoming dangerous. It can write text, but it cannot take over the pipeline contract.


In [ ]:
def validate_proposal(proposal: dict, metrics: dict) -> dict:
    """Validate the Manager Agent's proposal object before it can enter evaluation_report.json."""
    if not isinstance(proposal, dict):
        raise ValueError("LLM proposal must be a JSON object")

    missing = PROPOSAL_FIELDS - set(proposal)
    extra = set(proposal) - PROPOSAL_FIELDS
    if missing:
        raise ValueError(f"LLM proposal missing fields: {sorted(missing)}")
    if extra:
        raise ValueError(f"LLM proposal has unexpected fields: {sorted(extra)}")

    action = proposal["recommended_action"]
    if action not in {"retune", "proceed"}:
        raise ValueError("recommended_action must be retune or proceed")

    expected_action = "retune" if metrics["below_threshold"] else "proceed"
    if action != expected_action:
        raise ValueError(
            "recommended_action conflicts with deterministic threshold gate: "
            f"expected {expected_action}, got {action}"
        )

    focus_labels = proposal["focus_labels"]
    if not isinstance(focus_labels, list) or not focus_labels:
        raise ValueError("focus_labels must be a non-empty list")
    invalid_labels = [label for label in focus_labels if label not in LABELS]
    if invalid_labels:
        raise ValueError(f"focus_labels contains invalid labels: {invalid_labels}")

    if not isinstance(proposal["suggested_params"], dict):
        raise ValueError("suggested_params must be an object")
    if action == "proceed" and proposal["suggested_params"] != {}:
        raise ValueError("proceed proposals must not carry suggested_params")
    if action == "retune":
        params = proposal["suggested_params"]
        if not {"threshold", "max_length"} <= set(params):
            raise ValueError("retune proposals need threshold and max_length")
        if not isinstance(params["threshold"], (int, float)):
            raise ValueError("threshold must be numeric")
        if not isinstance(params["max_length"], int):
            raise ValueError("max_length must be an integer")
    if not isinstance(proposal["reason"], str) or not proposal["reason"].strip():
        raise ValueError("reason must be a non-empty string")
    if not isinstance(proposal["code_notes"], str):
        raise ValueError("code_notes must be a string")

    return proposal

## 11. Preparing a small LLM prompt

The evaluator agent does not send the whole classifier source or every misclassified id into the prompt. That would be too large and not very useful.

Instead, the LLM gets:

- compact metrics, including `class_accuracy` and `class_support`
- the selected `eval_split`
- a small classifier summary (`THRESHOLD`, `MAX_LENGTH`, `MODEL`, `MODEL_DIR`)
- up to 25 misclassified examples
- the deterministic proposal

The misclassified sample includes the headline, true label, predicted label, confidence, and probabilities. This gives the LLM enough context to describe a real failure pattern, for example if `up` headlines are often being predicted as `neutral`.

The prompt also explicitly explains the support rule: a class with support `0` is absent from this split, not collapsed. That matters because the LLM should not write scary collapse notes for a label that simply has no rows.

The LLM may only return two fields: `reason` and `code_notes`. Everything that controls the loop remains fixed by code.


In [ ]:
def _prompt_metrics(metrics: dict) -> dict:
    return {
        "accuracy": metrics["accuracy"],
        "below_threshold": metrics["below_threshold"],
        "class_accuracy": metrics["class_accuracy"],
        "class_support": metrics["class_support"],
        "misclassified_count": metrics["misclassified_count"],
    }


def _misclassified_sample(rows: list[dict], limit: int = MISCLASSIFIED_SAMPLE_SIZE) -> list[dict]:
    sample = []
    for row in rows:
        if row["label"] == row["predicted_label"]:
            continue
        sample.append({
            "headline": row["article_title"],
            "true_label": row["label"],
            "predicted_label": row["predicted_label"],
            "confidence": float(row["confidence"]),
            "prob_up": float(row["prob_up"]),
            "prob_down": float(row["prob_down"]),
            "prob_neutral": float(row["prob_neutral"]),
        })
        if len(sample) >= limit:
            break
    return sample


def _classifier_summary_for_prompt(code_text: str) -> dict:
    model = _find_assignment(code_text, "MODEL")
    return {
        "threshold": _find_assignment(code_text, "THRESHOLD"),
        "max_length": _find_assignment(code_text, "MAX_LENGTH"),
        "model": model,
        "model_dir": _find_assignment(code_text, "MODEL_DIR"),
        "uses_finbert": "finbert" in code_text.lower(),
        "maps_sentiment_to_label": "predicted_label" in code_text,
    }


## 12. Calling Ollama and falling back safely

This is the agentic part, but with guardrails.

If `EVALUATOR_USE_OLLAMA=true`, Sabina calls local Ollama. The LLM response must be JSON with exactly:

```json
{"reason": "...", "code_notes": "..."}
```

If the response is invalid, times out, or tries to return extra fields, the evaluator prints a small log line and uses the deterministic base proposal. This means the pipeline can still run offline.

This was important for the course demo because not everyone has the same local model setup, and the evaluator should not fail just because Ollama is not running.


In [ ]:
def _build_llm_prompt(
    metrics: dict,
    code_text: str,
    base_proposal: dict,
    failure_sample: list[dict],
) -> str:
    payload = {
        "eval_split": metrics.get("eval_split", "test"),
        "metrics": _prompt_metrics(metrics),
        "classifier_summary": _classifier_summary_for_prompt(code_text),
        "misclassified_sample": failure_sample,
        "deterministic_proposal": base_proposal,
    }
    return (
        "You are the evaluator agent in a multi-agent stock-move "
        "prediction pipeline.\n\n"
        "Review the deterministic metrics and classifier summary. The action, "
        "focus labels, and suggested params are already fixed by code and must "
        "not be changed by you.\n\n"
        "Return ONLY valid JSON with exactly these two string fields:\n"
        '{"reason": "...", "code_notes": "..."}\n\n'
        "Ground your reason in accuracy, target, weakest labels, and any useful "
        "classifier observation. Use the misclassified sample to describe the "
        "failure pattern when possible. Do not include markdown or extra keys.\n\n"
        "Judge per-class accuracy together with class_support, not just the "
        "aggregate: a classifier that predicts one class for almost everything "
        "can score near that class's share of the data while learning nothing. "
        "A class with support 0 is absent from this split, not collapsed. The "
        "eval_split field says which held-out split these metrics come from; "
        "retunes are scored on val so the test split stays unseen until the "
        "final report. Prefer observations that improve balance across classes "
        "over ones that chase the aggregate number.\n\n"
        f"INPUT:\n{json.dumps(payload, indent=2)}"
    )


def _ollama_generate(prompt: str) -> str:
    body = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": LLM_TEMPERATURE},
    }).encode("utf-8")

    request = urllib.request.Request(
        OLLAMA_URL,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=OLLAMA_TIMEOUT_SECONDS) as response:
        payload = json.loads(response.read().decode("utf-8"))
    return payload.get("response", "")


## 12b. Parsing and validating the LLM response

Ollama sometimes returns clean JSON, but sometimes a model wraps JSON in Markdown fences or adds a little text around it. `_extract_json_object` is a small tolerance layer for that.

After parsing, `_validate_llm_review` is strict again. It only accepts exactly two string fields:

- `reason`
- `code_notes`

If the model tries to return `recommended_action`, `focus_labels`, or `suggested_params`, the response is rejected and the deterministic fallback is used. This was one of the most important safeguards in the final version.

In [ ]:
def _extract_json_object(text: str) -> dict:
    # Parse plain JSON or a fenced JSON object returned by an LLM.
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = re.sub(r"^```(?:json)?\s*", "", stripped, flags=re.I)
        stripped = re.sub(r"\s*```$", "", stripped)
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", stripped, re.S)
        if not match:
            raise
        return json.loads(match.group(0))


def _validate_llm_review(review: dict) -> dict:
    # Accept only the fields the LLM is allowed to write.
    if not isinstance(review, dict):
        raise ValueError("LLM review must be a JSON object")
    if set(review) != {"reason", "code_notes"}:
        raise ValueError("LLM review may only contain reason and code_notes")
    if not isinstance(review["reason"], str) or not review["reason"].strip():
        raise ValueError("LLM reason must be a non-empty string")
    if not isinstance(review["code_notes"], str):
        raise ValueError("LLM code_notes must be a string")
    return {
        "reason": review["reason"].strip(),
        "code_notes": review["code_notes"].strip(),
    }

## 13. Applying the LLM review

This function is where the base proposal and LLM output meet.

The logic is:

1. If no LLM is configured, return the base proposal.
2. Build the prompt.
3. Call either the injected test LLM function or real Ollama.
4. Parse the JSON response.
5. Accept only `reason` and `code_notes`.
6. Merge those two text fields into the base proposal.
7. Validate the final proposal again.

So even when Ollama is on, the final action and retune params still come from deterministic code.

In [ ]:
def apply_llm_review(
    metrics: dict,
    code_text: str,
    base_proposal: dict,
    failure_sample: list[dict],
    llm_fn: Callable[[str], str] | None = None,
) -> dict:
    # Let an LLM improve reason/code_notes without changing control fields.
    if llm_fn is None and not USE_OLLAMA:
        return base_proposal

    prompt = _build_llm_prompt(metrics, code_text, base_proposal, failure_sample)
    try:
        response = (llm_fn or _ollama_generate)(prompt)
        review = _validate_llm_review(_extract_json_object(response))
    except (ValueError, json.JSONDecodeError, TimeoutError, OSError) as error:
        print(f"[sabina] LLM review failed ({error}); using deterministic fallback.")
        return base_proposal

    proposal = {
        **base_proposal,
        "reason": review["reason"],
        "code_notes": review["code_notes"],
    }
    return validate_proposal(proposal, metrics)

## 14. Building the final report

`build_report` is the core of the evaluator. It connects all helper functions in the actual order the agent uses them:

1. select the requested split (`val` or `test`)
2. validate the selected prediction rows
3. compute metrics, including `class_support`
4. statically review the classifier code and metrics
5. build the deterministic base proposal
6. collect a small misclassified sample
7. optionally let the LLM improve the text fields
8. return the full `evaluation_report.json` object

The split selection happens before validation and scoring. That is deliberate: `predictions_test.csv` now contains both `val` and `test` held-out rows, and Sabina must score only one of them at a time.

This function is also easy to test because it accepts rows and code text directly. Tests can pass a fake `llm_fn`, so they do not need live Ollama or network calls.


In [ ]:
def _select_split(rows: list[dict], eval_split: str) -> list[dict]:
    if eval_split not in ("val", "test"):
        raise ValueError(f"eval_split must be 'val' or 'test', got {eval_split!r}")

    selected = [row for row in rows if row.get("split") == eval_split]
    if not selected:
        raise ValueError(
            f"no split={eval_split} rows in predictions — regenerate "
            "processed_data.csv with the Processing agent (train/val/test split)"
        )
    return selected


def build_report(
    rows: list[dict],
    code_text: str,
    llm_fn: Callable[[str], str] | None = None,
    eval_split: str = "test",
) -> dict:
    rows = _select_split(rows, eval_split)
    validate_predictions(rows)

    metrics = {**compute_metrics(rows), "eval_split": eval_split}
    code_notes = review_classifier_metrics(
        code_text,
        metrics["class_accuracy"],
        metrics["class_support"],
    )

    base_proposal = validate_proposal(
        make_base_proposal(metrics, code_text, code_notes),
        metrics,
    )

    failure_sample = _misclassified_sample(rows)
    proposal = apply_llm_review(
        metrics,
        code_text,
        base_proposal,
        failure_sample,
        llm_fn=llm_fn,
    )

    return {**metrics, "proposal": proposal}


## 15. LangGraph nodes

The graph is intentionally boring: load, evaluate, write.

That is a good thing here. Sabina should not branch, mutate other agents' files, or make orchestration decisions. The only extra state that matters is `eval_split`, which is passed into `build_report` so the same evaluator can serve two moments in the pipeline:

- `eval_split="val"` while Jack is still retuning
- `eval_split="test"` once Jack has selected the best classifier and wants the honest final number


In [ ]:
def _write_json(path: str, obj: dict) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def load_inputs(state: EvaluatorState) -> dict:
    return {
        "predictions": _read_predictions(state["predictions_path"]),
        "code_text": _read_code(state["classifier_code_path"]),
    }


def evaluate(state: EvaluatorState) -> dict:
    return {
        "report": build_report(
            state["predictions"],
            state["code_text"],
            eval_split=state["eval_split"],
        )
    }


def write_report(state: EvaluatorState) -> dict:
    output_path = state.get("output_path") or os.path.join(OUTPUT_DIR, "evaluation_report.json")
    _write_json(output_path, state["report"])
    return {"output_path": output_path}


## 16. EvaluatorAgent class and test entry point

`EvaluatorAgent` is the small wrapper that gives Sabina the same `.run()` interface as the other agents.

The important API detail is the optional `eval_split` argument:

```python
EvaluatorAgent().run(predictions=..., classifier_code=..., eval_split="val")
```

Jack's pipeline uses `val` during the retune loop and `test` for the final scoring pass. If someone runs Sabina directly from the command line, the default is still `test`, because that is the most intuitive standalone behavior.


In [ ]:
def build_graph(checkpointer):
    from langgraph.graph import StateGraph, START, END

    builder = StateGraph(EvaluatorState)

    builder.add_node("load_inputs", load_inputs)
    builder.add_node("evaluate", evaluate)
    builder.add_node("write_report", write_report)

    builder.add_edge(START, "load_inputs")
    builder.add_edge("load_inputs", "evaluate")
    builder.add_edge("evaluate", "write_report")
    builder.add_edge("write_report", END)

    return builder.compile(checkpointer=checkpointer)


class EvaluatorAgent(Agent):
    def __init__(self, *, output_dir=OUTPUT_DIR, checkpointer=None, thread_id="evaluator"):
        self._output_dir = output_dir
        super().__init__(checkpointer=checkpointer, thread_id=thread_id)

    def build_graph(self, checkpointer):
        return build_graph(checkpointer)

    def run(self, predictions: str, classifier_code: str, eval_split: str = "test") -> dict:
        output_path = os.path.join(self._output_dir, "evaluation_report.json")
        return self._invoke({
            "predictions_path": predictions,
            "classifier_code_path": classifier_code,
            "output_path": output_path,
            "eval_split": eval_split,
        })


if __name__ == "__main__":
    agent = EvaluatorAgent()
    state = agent.run(
        predictions="mock_data/predictions_test.csv",
        classifier_code="mock_data/classifier.py",
    )
    print("Output file:", state["output_path"])


## 17. How I would explain the final design

My main goal was to make the evaluator useful without making it risky. Jack's retune loop depends on my report, so the report has to be boringly trustworthy.

The final design separates the responsibilities:

- deterministic code selects the split (`val` during retuning, `test` for the final score)
- deterministic code validates the prediction file against the shared contract
- deterministic code calculates accuracy, per-class accuracy, support, and wrong ids
- deterministic code decides whether the report is below threshold
- deterministic code chooses `focus_labels` and `suggested_params`
- the LLM only improves the human-readable explanation fields

The biggest recent fix is `class_support`. A class accuracy of `0.0` used to mean two different things: either the classifier completely failed on that label, or the selected split had no rows for that label. Those are not the same. Now Sabina reports support explicitly, so Jack can block real class collapse without punishing a split that simply has no examples of one class.

For the demo, Ollama can be enabled like this:

```bash
EVALUATOR_USE_OLLAMA=true EVALUATOR_OLLAMA_MODEL=llama3.2 uv run main.py
```

If Ollama is not running, the Evaluator Agent prints a fallback message and still writes a valid `evaluation_report.json`.

The short version is: Sabina grades Nadi's classifier, reports enough context for Jack to make a fair retune decision, and lets the LLM help with wording without giving it control over the pipeline.
